# TP-MCTS Experiments

Two experiment scripts, run from this notebook.

| Script | Purpose |
|--------|---------|
| `run_mcts_heuristic_comparison.py` | TP-MCTS score across 4 NASA scenarios × 6 heuristics |
| `run_heuristic_runtime_per_call.py` | Per-call timing (wrapper + worker + cache hit/miss) |

**Setup:** clone the repo and `cd` into it first (same as `demo.ipynb` cells 1-4).

In [1]:
# Clone repo (skip if already done in this Colab session)
import os

if not os.path.exists('/content/tp_mcts'):
    %cd /content
    !git clone https://github.com/eliezerRevach/tp_mcts.git

%cd /content/tp_mcts
!pip -q install dill numpy pandas openpyxl
print('Ready:', os.getcwd())

/content
Cloning into 'tp_mcts'...
remote: Enumerating objects: 2585, done.
remote: Counting objects: 100% (614/614), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 2585 (delta 555), reused 545 (delta 502), pack-reused 1971 (from 1)
Receiving objects: 100% (2585/2585), 15.66 MiB | 8.85 MiB/s, done.
Resolving deltas: 100% (1575/1575), done.
/content/tp_mcts
Ready: /content/tp_mcts


## Config

Edit the values below, then run the cells for each experiment.

In [ ]:
# ── Shared config ─────────────────────────────────────────────────────────
from pathlib import Path
import itertools

def _find_repo_root() -> Path:
    """Locate repo root (folder containing scripts/run_mcts_heuristic_comparison.py)."""
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "run_mcts_heuristic_comparison.py").is_file():
            return p
    return start

REPO_ROOT = _find_repo_root()

RUNS                 = 10     # runs per (scenario × heuristic)
SEED                 = 123    # random seed — same seed across all runs
SEARCH_TIME          = 1      # MCTS search time per step (seconds)
EXPLORATION_CONSTANT = 0.25  # UCT exploration constant C
REWARD_MODE          = "deadline"  # "deadline" or "terminal"
DOMAIN               = "machinery"  # must match `domains` in unified_planning/run_domain.py
DISCOUNT_FACTOR      = 1         # MDP gamma (MCTS discounted backup); also --gamma on CLI
STEP_PENALTY         = 0        # added to reward each transition (MDP.step); discourage long plans

# Scenario grid — Script 1 only
OBJECTS      = [2]      # object_amount values
DEADLINES    = [35]    # deadline values

# Heuristics — used by both scripts
# Options: ptrpg_old  baseline  baseline_cached  atomic_exact  atomic_exact_cached  fast_atom_cache
# HEURISTICS   = ["ptrpg_old", "baseline", "baseline_cached", "atomic_exact", "atomic_exact_cached", "fast_atom_cache"]
HEURISTICS   = ["atomic_exact"]

# Runtime benchmark grid — Script 2 (same lists as Script 1 unless you override)
RT_OBJECTS   = OBJECTS
RT_DEADLINES = DEADLINES
RT_MAX_STEPS = 1000
# Per-scenario heuristic depth = deadline (see Script 2 run cell)
RT_SCENARIO_GRID = list(itertools.product(RT_OBJECTS, RT_DEADLINES))

# Output paths — absolute so download / pandas always match where scripts write
MCTS_CSV     = str((REPO_ROOT / "results" / "mcts_heuristic_comparison.csv").resolve())
RUNTIME_CSV  = str((REPO_ROOT / "results" / "heuristic_runtime_per_call.csv").resolve())
RUNTIME_XLSX = str((REPO_ROOT / "results" / "heuristic_runtime_per_call.xlsx").resolve())

print("Config:")
print(f"  REPO_ROOT      : {REPO_ROOT}")
print(f"  MCTS scenarios : domain={DOMAIN}  objects={OBJECTS}  deadlines={DEADLINES}  runs={RUNS}  seed={SEED}  C={EXPLORATION_CONSTANT}  gamma={DISCOUNT_FACTOR}  step_penalty={STEP_PENALTY}")
print(f"  Heuristics     : {HEURISTICS}")
print(f"  Runtime bench  : {DOMAIN}  grid={RT_SCENARIO_GRID}  max_steps={RT_MAX_STEPS}  (depth=deadline per scenario)")
print(f"  MCTS_CSV       : {MCTS_CSV}")
print(f"  RUNTIME_CSV    : {RUNTIME_CSV}")
print(f"  RUNTIME_XLSX   : {RUNTIME_XLSX}")

Config:
  REPO_ROOT      : /content/tp_mcts
  MCTS scenarios : objects=[2, 3]  deadlines=[35]  runs=20  seed=123  C=0.05
  Heuristics     : ['ptrpg_old', 'baseline', 'baseline_cached', 'atomic_exact', 'atomic_exact_cached', 'fast_atom_cache']
  Runtime bench  : nasa_rover  grid=[(2, 35), (3, 35)]  max_steps=1000  (depth=deadline per scenario)
  MCTS_CSV       : /content/tp_mcts/results/mcts_heuristic_comparison.csv
  RUNTIME_CSV    : /content/tp_mcts/results/heuristic_runtime_per_call.csv
  RUNTIME_XLSX   : /content/tp_mcts/results/heuristic_runtime_per_call.xlsx


## Script 1 — MCTS Heuristic Comparison

Runs TP-MCTS on each **(object_amount, deadline) × heuristic** combination.

- Results are saved **incrementally** — partial data survives a Colab timeout.
- Output: `results/mcts_heuristic_comparison.csv`

In [ ]:
import subprocess

_h   = " ".join(HEURISTICS)
_obj = " ".join(str(o) for o in OBJECTS)
_dl  = " ".join(str(d) for d in DEADLINES)

cmd = [
    "python", "scripts/run_mcts_heuristic_comparison.py",
    "--runs",                 str(RUNS),
    "--seed",                 str(SEED),
    "--search_time",          str(SEARCH_TIME),
    "--exploration_constant", str(EXPLORATION_CONSTANT),
    "--reward_mode",          REWARD_MODE,
    "--domain",               DOMAIN,
    "--discount_factor",      str(DISCOUNT_FACTOR),
    "--step_penalty",         str(STEP_PENALTY),
    "--output",               MCTS_CSV,
    "--objects",              *[str(o) for o in OBJECTS],
    "--deadlines",            *[str(d) for d in DEADLINES],
    "--heuristics",           *HEURISTICS,
]

print("Running:", " ".join(cmd), flush=True)
print()

# Stream output live — cwd=REPO_ROOT so relative paths match this notebook
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(REPO_ROOT),
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

Running: python scripts/run_mcts_heuristic_comparison.py --runs 20 --seed 123 --search_time 1 --exploration_constant 0.05 --reward_mode deadline --output /content/tp_mcts/results/mcts_heuristic_comparison.csv --objects 2 3 --deadlines 25 35 --heuristics ptrpg_old baseline baseline_cached atomic_exact atomic_exact_cached


  TP-MCTS Heuristic Comparison
  Domain    : nasa_rover
  Scenarios : [(2, 25), (2, 35), (3, 25), (3, 35)]
  Heuristics: ['ptrpg_old', 'baseline', 'baseline_cached', 'atomic_exact', 'atomic_exact_cached']
  Runs/exp  : 20  seed=123  C=0.05  reward_mode=deadline
  Total     : 20 experiments
  Output    : /content/tp_mcts/results/mcts_heuristic_comparison.csv

[1/20]

────────────────────────────────────────────────────────────
  [nasa_rover obj=2 dl=25] heuristic=ptrpg_old
  depth=25  runs=20  seed=123  C=0.05  reward_mode=deadline
────────────────────────────────────────────────────────────
  => success=8/20  rate=0.4  avg_time=22.75  wall=2012.5s
Wrote /content/tp_mc

: 

In [ ]:
import pandas as pd

df = pd.read_csv(MCTS_CSV)

# Pivot: rows = (objects, deadline), columns = heuristic, values = success_rate
pivot = df.pivot_table(
    index=["object_amount", "deadline"],
    columns="heuristic",
    values="success_rate",
    aggfunc="first",
)
print("=== Success rate by scenario × heuristic ===")
print(pivot.to_string())

print("\n=== Full results table ===")
cols = ["domain", "object_amount", "deadline", "heuristic", "discount_factor", "step_penalty",
        "amount_success", "success_rate", "avg_success_time", "std_success_time"]
cols = [c for c in cols if c in df.columns]
print(df[cols].to_string(index=False))

## Script 2 — Heuristic Per-Call Runtime Benchmark

For each **(object_amount, deadline)** in the configured grid, runs `greedy_parallel` for each heuristic and measures:

- `wrapper_avg_call_sec` — total heuristic call cost (includes STN work)
- `worker_avg_call_sec` — pure propagation cost
- `worker_cache_hit_avg_sec` / `worker_cache_miss_avg_sec` — cache breakdown

Outputs (merged across scenarios): `results/heuristic_runtime_per_call.csv`, and `results/heuristic_runtime_per_call.xlsx` with blank rows between scenarios (written in the results cell below).

In [3]:
import subprocess
import pandas as pd

results_dir = REPO_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
part_paths = []
frames = []

for idx, (obj, dl) in enumerate(RT_SCENARIO_GRID, start=1):
    part_csv = results_dir / f"_runtime_part_{obj}_{dl}.csv"
    cmd = [
        "python", "scripts/run_heuristic_runtime_per_call.py",
        "--domain",         DOMAIN,
        "--object_amount",  str(obj),
        "--deadline",       str(dl),
        "--heuristic_depth",str(dl),
        "--max_steps",      str(RT_MAX_STEPS),
        "--seed",           str(SEED),
        "--reward_mode",    REWARD_MODE,
        "--discount_factor",str(DISCOUNT_FACTOR),
        "--step_penalty",   str(STEP_PENALTY),
        "--output",         str(part_csv.resolve()),
        "--heuristics",     *HEURISTICS,
    ]
    print(f"\n{'='*60}")
    print(f"  Runtime benchmark  [{idx}/{len(RT_SCENARIO_GRID)}]  obj={obj}  deadline={dl}  depth={dl}")
    print(f"{'='*60}")
    print("Running:", " ".join(cmd), flush=True)
    print()

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(REPO_ROOT),
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    print(f"\nProcess exited with code {proc.returncode}")

    if proc.returncode != 0:
        raise RuntimeError(
            f"run_heuristic_runtime_per_call failed for obj={obj} deadline={dl} (exit {proc.returncode})"
        )

    part_paths.append(part_csv)
    frames.append(pd.read_csv(part_csv))

df_rt_merged = pd.concat(frames, ignore_index=True)
df_rt_merged.to_csv(RUNTIME_CSV, index=False)
print(f"\nMerged {len(frames)} scenario(s) -> {RUNTIME_CSV}  ({len(df_rt_merged)} rows)")

for p in part_paths:
    try:
        p.unlink()
    except OSError:
        pass


  Runtime benchmark  [1/2]  obj=2  deadline=35  depth=35
Running: python scripts/run_heuristic_runtime_per_call.py --domain nasa_rover --object_amount 2 --deadline 35 --heuristic_depth 35 --max_steps 1000 --seed 123 --output /content/tp_mcts/results/_runtime_part_2_35.csv --heuristics ptrpg_old baseline baseline_cached atomic_exact atomic_exact_cached fast_atom_cache


  Heuristic Per-Call Runtime Benchmark
  Scenario  : nasa_rover  obj=2  deadline=35
  H-depth   : 35  max_steps=1000  seed=123
  Heuristics: ['ptrpg_old', 'baseline', 'baseline_cached', 'atomic_exact', 'atomic_exact_cached', 'fast_atom_cache']
  Output    : /content/tp_mcts/results/_runtime_part_2_35.csv

  Heuristic : ptrpg_old  (ptrpg_old (trpg))
  Internal  : heuristic_name=trpg  strategy=baseline
  Scenario  : nasa_rover obj=2  deadline=35  depth=35
started step 0
Current state is state: store_of(s0, r0) ; store_of(s1, r0) ; on_board(c0, r0) ; hand_of(h0, r0) ; hand_of(h1, r0) ; store_of(s2, r1) ; store_of(s3, r1) ;

In [4]:
import pandas as pd

df_rt = pd.read_csv(RUNTIME_CSV)

timing_cols = [
    "heuristic",
    "wrapper_avg_call_sec",
    "worker_avg_call_sec",
    "worker_cache_hit_avg_sec",
    "worker_cache_miss_avg_sec",
    "worker_cache_hits",
    "worker_cache_misses",
    "plan_success",
]

spaced_parts = []
for i, (obj, dl) in enumerate(RT_SCENARIO_GRID):
    mask = (df_rt["object_amount"] == obj) & (df_rt["deadline"] == dl)
    df_grp = df_rt.loc[mask, timing_cols].sort_values("wrapper_avg_call_sec")
    print(f"=== obj={obj}  deadline={dl}  (fastest → slowest) ===")
    print(df_grp.to_string(index=False))
    print()
    spaced_parts.append(df_grp)
    if i < len(RT_SCENARIO_GRID) - 1:
        spaced_parts.append(pd.DataFrame([{c: float("nan") for c in timing_cols}]))

df_runtime_xlsx = pd.concat(spaced_parts, ignore_index=True)
with pd.ExcelWriter(RUNTIME_XLSX, engine="openpyxl") as writer:
    df_runtime_xlsx.to_excel(writer, sheet_name="runtime_ranking", index=False)
print(f"Wrote Excel (blank rows between scenarios): {RUNTIME_XLSX}")

=== obj=2  deadline=35  (fastest → slowest) ===
          heuristic  wrapper_avg_call_sec  worker_avg_call_sec  worker_cache_hit_avg_sec  worker_cache_miss_avg_sec  worker_cache_hits  worker_cache_misses  plan_success
atomic_exact_cached              0.027728             0.027662                  0.000034                   0.031741                 84                  569          True
          ptrpg_old              0.028856                  NaN                       NaN                        NaN                  0                    0          True
       atomic_exact              0.032626             0.032567                  0.000026                   0.037371                 84                  569          True
    fast_atom_cache              0.037240             0.037177                  0.000036                   0.042659                 84                  569          True
    baseline_cached              0.039620             0.039510                  0.000034              

## Download Results to your PC

Run this cell after any experiment.

- **On Colab**: Colab cannot write to a path on your PC. This cell triggers a **browser download** (usually to **Downloads**). To land files in `TP_MCTS\results` on Windows, either move them after download, or set your browser’s default download folder to that directory (Chrome: Settings → Downloads → Location).
- **Local (this repo on your machine)**: copies each CSV into `LOCAL_PC_RESULTS_DIR` (default: `C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results`). Override with env var `TP_MCTS_RESULTS_DIR`.

Re-run the **Config** cell first so `MCTS_CSV`, `RUNTIME_CSV`, and `RUNTIME_XLSX` are absolute paths under `REPO_ROOT`. If Script 1 was never run in this session, the MCTS CSV is skipped until you run it.

In [ ]:
from pathlib import Path
import os
import shutil


def _find_repo_root_dl() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "run_mcts_heuristic_comparison.py").is_file():
            return p
    return start


# Local copy destination (not Colab). Override with env TP_MCTS_RESULTS_DIR.
if "TP_MCTS_RESULTS_DIR" in os.environ:
    LOCAL_PC_RESULTS_DIR = Path(os.environ["TP_MCTS_RESULTS_DIR"])
elif os.name == "nt":
    LOCAL_PC_RESULTS_DIR = Path(r"C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results")
else:
    LOCAL_PC_RESULTS_DIR = _find_repo_root_dl() / "results"
# Hint on Colab (Linux VM): where to put files on your PC; override with TP_MCTS_RESULTS_DIR.
_PC_RESULTS_HINT = (
    Path(os.environ["TP_MCTS_RESULTS_DIR"])
    if "TP_MCTS_RESULTS_DIR" in os.environ
    else Path(r"C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results")
)


def _resolve_csv(path: str) -> Path | None:
    """Resolve CSV path even if kernel cwd != repo root (Colab / multi-root)."""
    p = Path(path)
    if p.is_file():
        return p
    alt = _find_repo_root_dl() / "results" / p.name
    if alt.is_file():
        return alt
    return None


def _is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def download_result(path: str) -> None:
    """Colab: browser download. Local: copy into LOCAL_PC_RESULTS_DIR."""
    p = _resolve_csv(path)
    if p is None:
        print(f"  [skip] not found: {path}")
        print(f"          tried: {Path(path).resolve()} and {_find_repo_root_dl() / 'results' / Path(path).name}")
        return
    sp = str(p.resolve())
    size_kb = p.stat().st_size / 1024

    if _is_colab():
        from google.colab import files

        files.download(sp)
        print(f"  [browser download] {sp}  ({size_kb:.1f} KB)")
        print(f"      → On your PC, move the file to: {_PC_RESULTS_HINT}")
        print("      (Or set Chrome/Edge default download folder to that path.)")
        return

    LOCAL_PC_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    dest = LOCAL_PC_RESULTS_DIR / p.name
    shutil.copy2(p, dest)
    print(f"  [copied] {sp}  ({size_kb:.1f} KB)")
    print(f"      → {dest.resolve()}")


print("Exporting results...")
download_result(MCTS_CSV)
download_result(RUNTIME_CSV)
download_result(RUNTIME_XLSX)
print("Done.")

Exporting results...
  [skip] not found: /content/tp_mcts/results/mcts_heuristic_comparison.csv
          tried: /content/tp_mcts/results/mcts_heuristic_comparison.csv and /content/tp_mcts/results/mcts_heuristic_comparison.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  [browser download] /content/tp_mcts/results/heuristic_runtime_per_call.csv  (2.6 KB)
      → On your PC, move the file to: C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results
      (Or set Chrome/Edge default download folder to that path.)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  [browser download] /content/tp_mcts/results/heuristic_runtime_per_call.xlsx  (5.5 KB)
      → On your PC, move the file to: C:\Users\eliezer\Documents\hw2_exam\TP_MCTS\results
      (Or set Chrome/Edge default download folder to that path.)
Done.


### If you cannot find results (Colab)

- **`/content` is erased** when the runtime restarts or disconnects. If you open Colab later without re-running the experiment cells, **`tp_mcts/results` will be empty** — there is nothing to download.
- **Downloads**: the file may be blocked, or saved under another browser profile / “Ask where to save” folder.

**Run the next cell** in the same session **after** experiments: it checks whether the CSVs exist, lists `results/`, and **shows the tables inside the notebook** (works even when download fails).

**To keep files across sessions**, mount Google Drive and copy `results/` there (see comments in that cell), or download from the **Files** sidebar: `content` → `tp_mcts` → `results` → right‑click → **Download**.

In [6]:
# Locate results + show CSVs in the notebook (works on Colab without using Downloads).
from pathlib import Path

try:
    _rr = REPO_ROOT
except NameError:
    raise RuntimeError(
        "Run the **Config** cell first (defines REPO_ROOT, MCTS_CSV, RUNTIME_CSV, RUNTIME_XLSX)."
    ) from None

results_dir = Path(REPO_ROOT) / "results"
print(f"REPO_ROOT     : {REPO_ROOT}")
print(f"results folder: {results_dir}  (exists={results_dir.is_dir()})")
print()

for label, path_str in [
    ("MCTS_CSV", MCTS_CSV),
    ("RUNTIME_CSV", RUNTIME_CSV),
    ("RUNTIME_XLSX", RUNTIME_XLSX),
]:
    p = Path(path_str)
    st = "OK " if p.is_file() else "MISSING — run the experiment script cell in this session"
    print(f"{st}  {label}: {p}")

if results_dir.is_dir():
    print("\nFiles in results/:")
    for f in sorted(results_dir.iterdir()):
        if f.is_file():
            print(f"  {f.name}  ({f.stat().st_size} bytes)")

import pandas as pd
from IPython.display import display

for label, path_str in [("MCTS comparison", MCTS_CSV), ("Heuristic runtime", RUNTIME_CSV)]:
    p = Path(path_str)
    if p.is_file():
        df = pd.read_csv(p)
        print(f"\n=== {label}: {p.name} ({len(df)} rows) ===")
        display(df)
    else:
        print(f"\n=== {label}: skip (file not found) ===")

# --- Optional: persist on Google Drive (uncomment, run mount cell first) ---
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# dest = Path("/content/drive/MyDrive/Colab_TP_MCTS_results")
# dest.mkdir(parents=True, exist_ok=True)
# for path_str in [MCTS_CSV, RUNTIME_CSV, RUNTIME_XLSX]:
#     p = Path(path_str)
#     if p.is_file():
#         shutil.copy2(p, dest / p.name)
#         print("Copied to Drive:", dest / p.name)

REPO_ROOT     : /content/tp_mcts
results folder: /content/tp_mcts/results  (exists=True)

MISSING — run the experiment script cell in this session  MCTS_CSV: /content/tp_mcts/results/mcts_heuristic_comparison.csv
OK   RUNTIME_CSV: /content/tp_mcts/results/heuristic_runtime_per_call.csv
OK   RUNTIME_XLSX: /content/tp_mcts/results/heuristic_runtime_per_call.xlsx

Files in results/:
  heuristic_runtime_per_call.csv  (2628 bytes)
  heuristic_runtime_per_call.xlsx  (5632 bytes)

=== MCTS comparison: skip (file not found) ===

=== Heuristic runtime: heuristic_runtime_per_call.csv (12 rows) ===


,heuristic,heuristic_label,heuristic_name_internal,strategy_internal,domain,object_amount,deadline,seed,heuristic_depth,max_steps,...,wrapper_first_call_sec,wrapper_avg_call_sec,worker_total_calls,worker_total_time_sec,worker_first_call_sec,worker_avg_call_sec,worker_cache_hits,worker_cache_misses,worker_cache_hit_avg_sec,worker_cache_miss_avg_sec
0,ptrpg_old,ptrpg_old (trpg),trpg,NaN,nasa_rover,2,35,123,35,1000,...,0.027180,0.028856,0,NaN,NaN,NaN,0,0,NaN,NaN
1,baseline,baseline,temporal_probabilistic_rpg,baseline,nasa_rover,2,35,123,35,1000,...,0.175048,0.071215,653,46.429519,0.114626,0.071102,84,569,0.000034,0.081593
2,baseline_cached,baseline_cached,temporal_probabilistic_rpg,baseline_cached,nasa_rover,2,35,123,35,1000,...,0.245109,0.039620,653,25.800280,0.186784,0.039510,84,569,0.000034,0.045338
3,atomic_exact,atomic_exact,temporal_probabilistic_rpg,atom_backtrack_exact,nasa_rover,2,35,123,35,1000,...,0.095496,0.032626,653,21.266238,0.068499,0.032567,84,569,0.000026,0.037371
4,atomic_exact_cached,atomic_exact_cached,temporal_probabilistic_rpg,atom_backtrack_cached,nasa_rover,2,35,123,35,1000,...,0.146084,0.027728,653,18.063523,0.117448,0.027662,84,569,0.000034,0.031741
5,fast_atom_cache,fast_atom_cache,temporal_probabilistic_rpg,fast_atom_cache,nasa_rover,2,35,123,35,1000,...,0.118208,0.037240,653,24.276265,0.089934,0.037177,84,569,0.000036,0.042659
6,ptrpg_old,ptrpg_old (trpg),trpg,NaN,nasa_rover,3,35,123,35,1000,...,0.049114,0.062662,0,NaN,NaN,NaN,0,0,NaN,NaN
7,baseline,baseline,temporal_probabilistic_rpg,baseline,nasa_rover,3,35,123,35,1000,...,0.200047,0.116803,1588,185.390891,0.145108,0.116745,178,1410,0.000044,0.131477
8,baseline_cached,baseline_cached,temporal_probabilistic_rpg,baseline_cached,nasa_rover,3,35,123,35,1000,...,0.186831,0.051281,1588,81.349974,0.140527,0.051228,178,1410,0.000050,0.057689
9,atomic_exact,atomic_exact,temporal_probabilistic_rpg,atom_backtrack_exact,nasa_rover,3,35,123,35,1000,...,0.152838,0.047507,1588,75.360177,0.105532,0.047456,178,1410,0.000039,0.053442
